In [1]:
import pandas as pd, sqlite3

df = pd.read_csv("KaggleV2-May-2016.csv")
conn = sqlite3.connect("noshow.db")
df.to_sql("appointments_raw", conn, if_exists="replace", index=False)

print(df.shape)
print(df.columns.tolist())

(110527, 14)
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']


In [2]:
def sql(query):
    return pd.read_sql(query, conn)

In [3]:
sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT AppointmentID) AS unique_appointments,
       COUNT(DISTINCT PatientId) AS unique_patients
FROM appointments_raw
""")

,total_rows,unique_appointments,unique_patients
0,110527,110527,62299


In [4]:
sql("""
SELECT MIN(Age) AS min_age,
       MAX(Age) AS max_age,
       SUM(CASE WHEN Age < 0 THEN 1 ELSE 0 END) AS negative_ages,
       SUM(CASE WHEN date(AppointmentDay) < date(ScheduledDay) THEN 1 ELSE 0 END) AS appt_before_booking
FROM appointments_raw
""")

,min_age,max_age,negative_ages,appt_before_booking
0,-1,115,1,5


In [5]:
conn.execute("DROP TABLE IF EXISTS appointments_clean")
conn.execute("""
CREATE TABLE appointments_clean AS
SELECT *,
       CASE WHEN "No-show" = 'Yes' THEN 1 ELSE 0 END AS no_show,
       CAST(julianday(date(AppointmentDay)) - julianday(date(ScheduledDay)) AS INTEGER) AS lead_days
FROM appointments_raw
WHERE Age >= 0
  AND date(AppointmentDay) >= date(ScheduledDay)
""")
conn.commit()

sql("SELECT COUNT(*) AS clean_rows FROM appointments_clean")

,clean_rows
0,110521


In [6]:
sql("""
SELECT SMS_received,
       COUNT(*) AS appointments,
       SUM(no_show) AS no_shows,
       ROUND(100.0 * AVG(no_show), 1) AS no_show_rate_pct
FROM appointments_clean
GROUP BY SMS_received
""")

,SMS_received,appointments,no_shows,no_show_rate_pct
0,0,75039,12530,16.7
1,1,35482,9784,27.6


In [7]:
sql("""
SELECT SMS_received,
       ROUND(AVG(lead_days), 1) AS avg_lead_days,
       SUM(CASE WHEN lead_days = 0 THEN 1 ELSE 0 END) AS same_day_appts
FROM appointments_clean
GROUP BY SMS_received
""")

,SMS_received,avg_lead_days,same_day_appts
0,0,6.0,38562
1,1,19.0,0


In [8]:
sql("""
SELECT CASE
         WHEN lead_days = 0 THEN '0: same day'
         WHEN lead_days <= 2 THEN '1: 1-2 days'
         WHEN lead_days <= 7 THEN '2: 3-7 days'
         WHEN lead_days <= 14 THEN '3: 8-14 days'
         WHEN lead_days <= 30 THEN '4: 15-30 days'
         ELSE '5: 31+ days'
       END AS wait_group,
       SMS_received,
       COUNT(*) AS appointments,
       ROUND(100.0 * AVG(no_show), 1) AS no_show_rate_pct
FROM appointments_clean
GROUP BY wait_group, SMS_received
ORDER BY wait_group, SMS_received
""")

,wait_group,SMS_received,appointments,no_show_rate_pct
0,0: same day,0,38562,4.6
1,1: 1-2 days,0,11938,22.7
2,2: 3-7 days,0,8699,26.6
3,2: 3-7 days,1,11548,23.8
4,3: 8-14 days,0,5021,33.8
5,3: 8-14 days,1,7004,28.1
6,4: 15-30 days,0,6851,36.9
7,4: 15-30 days,1,10520,29.8
8,5: 31+ days,0,3968,37.5
9,5: 31+ days,1,6410,30.2


In [9]:
!pip install anthropic -q

import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

response = client.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say hello in exactly five words."}]
)
print(response.content[0].text)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.7 MB/s eta 0:00:00
Hello there, hope you're well!


In [11]:
SCHEMA = """
Table: appointments_clean (110,521 medical appointments, Brazil, 2016)
Columns:
- PatientId, AppointmentID
- Gender ('F' or 'M')
- ScheduledDay, AppointmentDay (text timestamps)
- Age (integer)
- Neighbourhood (text)
- Scholarship, Hipertension, Diabetes, Alcoholism (0/1)
- Handcap (0-4)
- SMS_received (1 = got a reminder text, 0 = no text)
- no_show (1 = missed the appointment, 0 = attended)
- lead_days (days between booking and appointment)
"""

SYSTEM = f"""You are a data analyst assistant for a patient no-show study.
Answer questions by writing SQLite queries with the run_sql tool, then explain the results in plain English.
{SCHEMA}
Rules:
- Only write SELECT queries.
- Report rates as percentages with group sizes.
- When discussing SMS effects, note that the data is observational and that texts were only sent for appointments booked 3+ days ahead.
"""

tools = [{
    "name": "run_sql",
    "description": "Run a read-only SQLite SELECT query on the appointments database and return the results.",
    "input_schema": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "A single SQLite SELECT statement"}},
        "required": ["query"]
    }
}]

def run_sql(query):
    if not query.strip().lower().startswith(("select", "with")):
        return "Error: only SELECT queries are allowed."
    try:
        return pd.read_sql(query, conn).head(50).to_string(index=False)
    except Exception as e:
        return f"SQL error: {e}"

def ask(question, show_sql=True, max_steps=8):
    messages = [{"role": "user", "content": question}]
    for _ in range(max_steps):
        response = client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=1500,
            system=SYSTEM,
            tools=tools,
            messages=messages
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            print("".join(b.text for b in response.content if b.type == "text"))
            return

        results = []
        for block in response.content:
            if block.type == "tool_use":
                query = block.input["query"]
                if show_sql:
                    print("SQL:\n" + query + "\n")

                # Execute the tool and fetch results
                tool_output = run_sql(query)
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": tool_output
                })

        # Append the tool results to messages to continue the conversation
        messages.append({"role": "user", "content": results})

In [12]:
ask("For patients over 60, what is the no-show rate for those who got a text vs. those who didn't?")

SQL:
SELECT SMS_received, COUNT(*) AS n, SUM(no_show) AS no_shows, ROUND(100.0*SUM(no_show)/COUNT(*),2) AS no_show_pct FROM appointments_clean WHERE Age > 60 GROUP BY SMS_received

SQL:
SELECT SMS_received, COUNT(*) AS n, SUM(no_show) AS no_shows, ROUND(100.0*SUM(no_show)/COUNT(*),2) AS no_show_pct FROM appointments_clean WHERE Age > 60 AND lead_days >= 3 GROUP BY SMS_received

**Patients over 60, all appointments**

| Got a text? | Appointments | No-shows | No-show rate |
|---|---|---|---|
| No | 13,629 | 1,826 | 13.40% |
| Yes | 6,132 | 1,179 | 19.23% |

On the raw numbers, patients who got a text missed more appointments (19.2% vs. 13.4%). That comparison is misleading. Texts were only sent for appointments booked 3 or more days ahead. Appointments booked further out tend to have higher no-show rates anyway. The "no text" group also includes many short-notice bookings, which are rarely missed.

**Like-for-like comparison: only appointments booked 3+ days ahead**

| Got a text? | App

In [13]:
sql("""
SELECT SMS_received,
       COUNT(*) AS n,
       ROUND(100.0 * AVG(no_show), 2) AS no_show_pct
FROM appointments_clean
WHERE Age > 60 AND lead_days >= 3
GROUP BY SMS_received
""")

,SMS_received,n,no_show_pct
0,0,4671,23.93
1,1,6132,19.23


In [14]:
ask("Which 5 neighbourhoods have the highest no-show rates? Only include neighbourhoods with at least 500 appointments.")

SQL:
SELECT Neighbourhood, COUNT(*) AS n, SUM(no_show) AS missed, ROUND(100.0*SUM(no_show)/COUNT(*),2) AS no_show_pct FROM appointments_clean GROUP BY Neighbourhood HAVING COUNT(*) >= 500 ORDER BY no_show_pct DESC LIMIT 5

Here are the five neighbourhoods with the highest no-show rates. Each has at least 500 appointments.

| Rank | Neighbourhood | Appointments | No-shows | No-show rate |
|---|---|---|---|---|
| 1 | Santos Dumont | 1,276 | 369 | 28.92% |
| 2 | Santa Clara | 506 | 134 | 26.48% |
| 3 | Itararé | 3,514 | 923 | 26.27% |
| 4 | Jesus de Nazareth | 2,853 | 696 | 24.40% |
| 5 | Ilha do Príncipe | 2,266 | 532 | 23.48% |

- **Overall comparison:** I didn't query the overall rate. The dataset is commonly around 20%, so these neighbourhoods look several points higher. Check that figure before relying on it.
- **Santa Clara:** It has only 506 appointments, just over the cutoff. Its rate is less stable than the others, because a small number of appointments could move it noticeably.


In [15]:
ask("Do patients with diabetes miss more appointments than patients without it?")

SQL:
SELECT Diabetes, COUNT(*) AS n, SUM(no_show) AS missed, ROUND(100.0*SUM(no_show)/COUNT(*),2) AS no_show_pct FROM appointments_clean GROUP BY Diabetes

**No. Patients with diabetes missed slightly fewer appointments than patients without it.**

| Diabetes | Appointments | Missed | No-show rate |
|---|---|---|---|
| No (0) | 102,578 | 20,884 | **20.36%** |
| Yes (1) | 7,943 | 1,430 | **18.00%** |

- The gap is about 2.4 percentage points. Diabetic patients' no-show rate is roughly 12% lower in relative terms.
- With nearly 8,000 diabetic appointments, a gap this size is probably not random noise. I didn't run a significance test.
- This is a raw comparison, and I haven't adjusted for other factors. Diabetic patients tend to be older, and older patients usually miss fewer appointments. Age could therefore explain some or all of the gap, rather than diabetes itself. I can run the comparison within age bands to check.
- The rows are appointments, not unique patients. Patients with seve

In [17]:
# Always calculate every number from the data with run_sql. Never use outside knowledge or assumptions about this dataset.

i’m analyzing 110K medical appointments to test whether SMS reminders reduce no-shows. The naive comparison suggested texts made no-shows worse, but I found that lead time was a confounder. Once I compared appointments with similar wait times, texted patients had 3–7 percentage points lower no-show rates. Next I’m testing significance and running a regression in R.